# Multi-turn pilot v4

Use a GPU runtime and check the actual CU/hour. Run cells in order, not Run all. This pilot uses one topic, two models, pressure and neutral, one conversation order; review results before extending. The model and history outputs are exploratory, not the final study. Download results before disconnecting.

In [ ]:
from google.colab import files
import os, zipfile
uploaded = files.upload()
assert "multiturn-pilot-v4.zip" in uploaded
with zipfile.ZipFile("multiturn-pilot-v4.zip") as z:
    z.extractall("/content/multiturn-pilot-v4")
os.chdir("/content/multiturn-pilot-v4")
print("Working directory:", os.getcwd())


In [ ]:
%pip -q install "transformers==5.22.1" accelerate numpy huggingface_hub


In [ ]:
from google.colab import userdata
from huggingface_hub import login, HfApi
login(token=userdata.get("HF_TOKEN"), add_to_git_credential=False)
import torch
assert torch.cuda.is_available(), "Select a GPU runtime"
print("GPU:", torch.cuda.get_device_name(0))
MODELS = {"llama": "meta-llama/Llama-3.1-8B-Instruct", "qwen": "Qwen/Qwen2.5-7B-Instruct"}
# Pinned to the v3 run's revisions (not "latest"), so v3 and v4 use the same weights.
REVISIONS = {"llama": "0e9e39f249a16976918f6564b8830bc894c89659",
             "qwen": "a09a35458c702b33eeacc393d103063234e8bc28"}
for name, repo in MODELS.items():
    latest = HfApi().model_info(repo).sha
    print(name, "pinned", REVISIONS[name], "| latest on Hub", latest, "(same)" if latest == REVISIONS[name] else "(differs; pinned is used)")


In [ ]:
import json, os, subprocess, sys
from pathlib import Path
for name, repo in MODELS.items():
    for condition in ("pressure", "neutral"):
        out = Path(f"results/{name}_four_day_{condition}_o1.json")
        if out.exists():
            prev = json.loads(out.read_text())
            if prev.get("status") == "complete" and prev["args"].get("revision") == REVISIONS[name]:
                print("Already complete:", out, flush=True)
                continue
            raise RuntimeError(f"Incomplete or mismatched result at {out}; download and inspect before rerunning")
        command = [sys.executable, "-u", "scripts/new_dialogue_pilot_v4.py", "--model", repo, "--revision", REVISIONS[name], "--device", "cuda", "--topic", "four_day_week", "--condition", condition, "--out", str(out)]
        print("Running", name, condition, flush=True)
        # A truncated history exits non-zero after saving; keep going with the other histories.
        code = subprocess.run(command).returncode
        status = json.loads(out.read_text()).get("status") if out.exists() else "no_output"
        print(name, condition, "exit", code, "status", status, flush=True)
print("Done. Any status other than 'complete' must be inspected before use.")


In [ ]:
import shutil
from google.colab import files
shutil.make_archive("/content/multiturn-pilot-v4-results", "zip", "/content/multiturn-pilot-v4", "results")
files.download("/content/multiturn-pilot-v4-results.zip")


Download and verify results before disconnecting and deleting the GPU runtime. The script persists output after each turn and readout but does not resume a partially completed history automatically.